# VoiceDoc Qwen3-0.6B fine-tuning
Attach reviewed SFT with genuine receipts, repository and pinned model. Select one GPU. Edit paths. Run cells in order.

In [ ]:
from pathlib import Path
import os, sys, subprocess
os.environ['CUDA_VISIBLE_DEVICES']='0'
REPO=Path('/kaggle/input/voicedoc-repo/VoiceDoc')  # edit
REVIEWED=Path('/kaggle/input/voicedoc-reviewed/reviewed-sft.jsonl')  # edit
MODEL=Path('/kaggle/input/qwen3-06b/snapshot')  # edit
REVISION='REPLACE_WITH_EXACT_40_CHAR_HF_COMMIT_SHA'
RUN=Path('/kaggle/working/voicedoc')
RESUME=None  # or same run checkpoint
assert (REPO/'ml/training/run_kaggle.py').is_file()
assert REVIEWED.is_file() and (MODEL/'config.json').is_file()


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'ml/training/requirements-kaggle.txt')],check=True)


In [ ]:
args=[sys.executable,str(REPO/'ml/training/run_kaggle.py'),'--reviewed-sft',str(REVIEWED),'--base-model-dir',str(MODEL),'--base-model-revision',REVISION,'--run-dir',str(RUN)]
subprocess.run(args+['--prepare-only'],cwd=REPO,check=True)


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==1
resume_args=['--resume-from-checkpoint',str(RESUME)] if RESUME else []
subprocess.run(args+resume_args,cwd=REPO,check=True)


In [ ]:
import json
print(json.dumps(json.loads((RUN/'training/run-metadata.json').read_text()),indent=2))
print('Merged model:',RUN/'merged')
print('Persist Kaggle output artifacts. Not approved for clinical deployment.')
